# A Quick Introduction to UNIX

**Advanced Computational Physics** &middot; Master in Physics &middot; USC

This notebook is an *interactive* guide: every command below can be run and modified.

## What you will learn

- What a shell is, and why you need one
- Moving around the file system
- Creating, copying, moving and removing files
- Looking inside files
- Wildcards, redirection and pipes
- Searching: `grep` and `find`
- Permissions, environment variables, processes
- Working on a remote machine
- Configuring your shell, and writing a first script

## Why the shell?

In physics you will meet machines you can only reach through a terminal:
computing clusters, remote servers, data acquisition systems.

- It is the **only** interface on most compute clusters.
- It is **scriptable**: anything you do by hand can be automated and repeated.
- It **composes**: small tools chain together to do large jobs.

A *shell* is the program that reads what you type and asks the operating system to do it.
The most common one is **bash**.

## Running shell commands from Jupyter

In a notebook, a line starting with `!` is sent to the shell:

```python
!ls
```

Each `!` line runs in **its own subshell**, which is why `!cd somewhere` has no lasting effect.
To change the notebook's own directory use the magic `%cd`.

In a real terminal you would type the command **without** the `!`.

## A sandbox to play in

Everything in this notebook happens inside a folder called `unix_sandbox`,
so that nothing else on your computer is touched. Running this cell again
starts from a clean state.

In [ ]:
import os, shutil

BASE = globals().get("BASE", os.getcwd())      # remember where we started
os.chdir(BASE)
shutil.rmtree("unix_sandbox", ignore_errors=True)
os.makedirs("unix_sandbox")
os.chdir("unix_sandbox")
print("Working inside:", os.path.basename(os.getcwd()) + "/")

## Where am I?

- `pwd` &mdash; *print working directory*
- `ls` &mdash; list the contents of a directory

In [ ]:
!pwd

In [ ]:
!ls -l

### Useful options for `ls`

| Option | Meaning |
|---|---|
| `-l` | long format: permissions, owner, size, date |
| `-h` | human-readable sizes (`4.0K`, `2.3M`) |
| `-a` | show hidden files (those whose name starts with `.`) |
| `-t` | sort by modification time, newest first |
| `-R` | descend into subdirectories |

Options can be combined: `ls -lha`.

In [ ]:
!ls -lha

## Paths

The file system is a **tree**. A path says how to reach a file in that tree.

| Path | Meaning |
|---|---|
| `/home/ana/data.txt` | **absolute**: starts at the root `/` |
| `data/run1.txt` | **relative**: starts at the current directory |
| `.` | the current directory |
| `..` | the parent directory |
| `~` | your home directory |

An absolute path always means the same file. A relative path depends on where you are.

In [ ]:
!echo "Home directory: $HOME"
!ls ..

### Moving around: `cd`

```bash
cd data          # go into ./data
cd ..            # go up one level
cd ~             # go home
cd -             # go back to the previous directory
cd               # with no argument, also goes home
```

Remember: inside Jupyter use `%cd`, because `!cd` only affects its own subshell.

## Getting help

| Command | What it does |
|---|---|
| `man ls` | the manual page for `ls` (press `q` to quit) |
| `ls --help` | short help (GNU/Linux only &mdash; see *POSIX*, later) |
| `apropos copy` | search the manuals by keyword |
| `which python3` | where does this command live? |
| `type cd` | is it a program, a builtin, an alias? |

`man` is a *pager*: it takes over the terminal, so it does not work well inside a notebook.

In [ ]:
!man ls | col -b | head -12

## Creating files and directories

- `mkdir name` &mdash; make a directory
- `mkdir -p a/b/c` &mdash; make the whole chain, and do not complain if it exists
- `touch file` &mdash; create an empty file (or update its timestamp)

In [ ]:
!mkdir -p data/raw
!touch data/raw/run1.txt data/raw/run2.txt
!ls -R data

## Removing files and directories

- `rm file` &mdash; remove a file
- `rmdir dir` &mdash; remove an **empty** directory
- `rm -r dir` &mdash; remove a directory and everything in it

> ⚠️ **There is no recycle bin.** `rm` is immediate and irreversible.
> Be especially careful with `rm -rf`, and never run it on a path you have not read twice.
> `rm -i` asks for confirmation on each file.

In [ ]:
!touch scratch.txt
!ls
!rm scratch.txt
!ls

## Copying and moving

- `cp source target` &mdash; copy
- `cp -r dir target` &mdash; copy a whole directory
- `mv source target` &mdash; move **or** rename (it is the same operation)

Both will **overwrite the target without asking**; `-i` makes them ask.

In [ ]:
!echo "Hello World"    > data/raw/run1.txt
!echo "This is a test"  > data/raw/run2.txt

!cp data/raw/run1.txt data/raw/run1_copy.txt
!mv data/raw/run2.txt data/raw/run2_renamed.txt
!ls data/raw

## Looking inside a file

| Command | What it shows |
|---|---|
| `cat file` | the whole file |
| `head -5 file` | the first 5 lines |
| `tail -5 file` | the last 5 lines |
| `tail -f file` | the end of the file, updating live (useful for logs) |
| `less file` | page through it (`q` to quit, `/` to search) |
| `wc -l file` | how many lines |

For a large file, never use `cat`: use `less`, `head` or `tail`.

In [ ]:
!cat data/raw/run1.txt

In [ ]:
# build a slightly larger file to play with
!seq 1 100 > numbers.txt
!head -3 numbers.txt
!echo "..."
!tail -3 numbers.txt
!wc -l numbers.txt

## Wildcards

The shell expands these **before** the command runs:

| Pattern | Matches |
|---|---|
| `*` | any number of characters (including none) |
| `?` | exactly one character |
| `[0-9]` | one character from the set |
| `{a,b}` | either alternative |

So `ls *.txt` lists every file ending in `.txt`.

In [ ]:
!ls data/raw/*.txt
!echo "---"
!ls data/raw/run?.txt

## Redirection

Every command has an input and two outputs. You can redirect them.

| Syntax | Meaning |
|---|---|
| `cmd > file` | send output to `file`, **overwriting** it |
| `cmd >> file` | **append** the output to `file` |
| `cmd < file` | read input from `file` |
| `cmd 2> file` | redirect error messages |
| `cmd > file 2>&1` | redirect output *and* errors to the same file |

In [ ]:
!echo "first line"  > log.txt
!echo "second line" >> log.txt
!cat log.txt

## Pipes

A pipe `|` sends the output of one command straight into the next.
This is the idea that makes the shell powerful: small tools, combined.

```bash
ls -l | wc -l                 # how many entries in this directory
cat data.txt | sort | uniq    # sorted, without repetitions
```

In [ ]:
!seq 1 20 | tail -5 | wc -l

In [ ]:
# a small pipeline: the 3 most frequent words
!printf "alpha\nbeta\nalpha\ngamma\nbeta\nalpha\n" > words.txt
!sort words.txt | uniq -c | sort -rn | head -3

## Searching *inside* files: `grep`

`grep PATTERN file` prints the lines that match.

| Option | Meaning |
|---|---|
| `-i` | ignore case |
| `-n` | show line numbers |
| `-r` | search recursively in a directory |
| `-v` | invert: show lines that do **not** match |
| `-c` | count matches instead of printing them |

In [ ]:
!grep -n "alpha" words.txt
!echo "---"
!grep -c "alpha" words.txt

## Searching *for* files: `find`

```bash
find . -name "*.txt"          # by name, from here down
find . -type d                # only directories
find . -name "*.log" -delete  # careful!
find . -mtime -1              # modified in the last day
```

In [ ]:
!find . -name "*.txt" 

## Permissions

`ls -l` shows something like `-rw-r--r--`:

```
 -    rw-      r--      r--
type  owner    group    others
```

- `r` read &middot; `w` write &middot; `x` execute (for a directory: *enter*)
- `chmod u+x script.sh` &mdash; give the **u**ser permission to e**x**ecute
- `chmod 644 file` &mdash; the same in octal: owner `rw-`, everyone else `r--`

You will need this the first time a script of yours refuses to run.

In [ ]:
!touch demo.sh
!ls -l demo.sh
!chmod u+x demo.sh
!ls -l demo.sh

## Environment variables

Variables the shell passes to every program it runs.

| Variable | Meaning |
|---|---|
| `$HOME` | your home directory |
| `$PATH` | where the shell looks for commands |
| `$SHELL` | which shell you are using |
| `$USER` | your login name |

```bash
export MY_DATA=/scratch/run42     # define one
echo $MY_DATA                     # use it
```

In [ ]:
!echo "Shell: $SHELL"
!echo "First three entries of PATH:"
!echo $PATH | tr ':' '\n' | head -3

In [ ]:
!which python3
!which ls

## Processes

| Command | What it does |
|---|---|
| `ps` | list your processes |
| `top` / `htop` | live view of what is running |
| `kill 1234` | ask process 1234 to stop |
| `kill -9 1234` | force it to stop |
| `cmd &` | run `cmd` in the background |
| `Ctrl-C` | interrupt the running command |
| `Ctrl-Z` then `bg` | suspend it, then resume in the background |

In [ ]:
!ps -o pid,comm | head -5

## Working on a remote machine

This is how you will use a computing cluster.

```bash
ssh user@server.usc.es              # open a remote shell
scp data.txt user@server:/scratch/  # copy a file there
scp user@server:/scratch/out.dat .  # and back
rsync -av data/ user@server:/data/  # copy only what changed
```

None of these can be demonstrated inside the notebook: they need a real terminal.

### Keeping a job alive: `screen` and `tmux`

If your connection drops, everything you were running dies with it &mdash; unless
you started it inside a `screen` or `tmux` session.

```bash
screen -S mysession    # start a named session
# ... launch your long job ...
Ctrl-a  d              # detach: the job keeps running
screen -r mysession    # reconnect later, from anywhere
screen -ls             # list your sessions
```

`tmux` does the same with a nicer interface; the detach key is `Ctrl-b d`.

## The common UNIX shells

| Shell | Strengths | Weaknesses | Best for |
|---|---|---|---|
| **sh** (Bourne) | Minimal, available everywhere | Very limited interactively | Legacy scripts |
| **bash** (Bourne Again) | Standard, POSIX-friendly, scripting workhorse | Lacks some modern conveniences | Teaching, scripting, servers |
| **zsh** (Z shell) | Smart completion, plugins, themes | Not always preinstalled | Daily work, power users |
| **ksh** (Korn) | POSIX compliant, stable | Less common nowadays | Legacy UNIX systems |
| **csh / tcsh** | C-like syntax, history | Poor for scripting, outdated | Historical interest |
| **fish** | Friendly, autosuggestions | Not POSIX compatible | Personal productivity |

Since macOS Catalina the default is **zsh**; most Linux distributions still default to **bash**.

**POSIX** (*Portable Operating System Interface*) is the standard that defines the
shell syntax and the utilities used throughout this notebook: stay within it and
your scripts run everywhere.

### What POSIX guarantees &mdash; and what it does not

POSIX (IEEE 1003) is the agreement that ended the divergence between the UNIX
variants of the 1980s. It fixes three things:

- the **C system calls** (`open`, `read`, `fork`, `kill`) &mdash; why `ps` and `Ctrl-C` behave alike everywhere;
- the **shell language**: `|`, `>`, `>>`, `2>&1`, `$VAR`, `for`, `if`;
- a **core set of utilities**: `ls`, `cp`, `grep`, `find`, `sed`, with a minimum set of options.

But the standard fixes the *minimum*, not the maximum, and every system adds its own
extensions. This is where macOS and Linux part company:

| Works on Linux | Fails on macOS | Why |
|---|---|---|
| `ls --help` | no such option | long options are a GNU extension |
| `sed -i 's/a/b/' f` | needs an argument: `sed -i '' ...` | BSD `sed` differs from GNU `sed` |

> **Rule of thumb.** Start scripts with `#!/bin/bash`, prefer short one-letter options,
> and distrust long ones such as `--color`. Everything taught in this notebook is
> inside that safe core.

In [ ]:
!echo $SHELL

## Configuring your shell

Your shell reads a configuration file every time it starts:

| Shell | File |
|---|---|
| bash | `~/.bashrc` (and `~/.bash_profile` on login) |
| zsh | `~/.zshrc` |

Typical contents:

```bash
alias ll='ls -lh'                        # a shortcut
alias ..='cd ..'
export MY_PROJECT=$HOME/projects/myapp   # a variable
export PATH=$HOME/bin:$PATH              # add your own programs
```

After editing it, run `source ~/.bashrc` to apply the changes to the current shell.

> ⚠️ **Do not let a notebook write into your `~/.bashrc`.**
> A cell that appends to it adds the same lines again every time you run it,
> and a broken `.bashrc` can leave you with a shell that will not start.
> Edit it by hand, with an editor, and keep a copy.

The cell below only **shows** the end of your configuration file; it changes nothing.

In [ ]:
import os

for name in ("~/.zshrc", "~/.bashrc"):
    path = os.path.expanduser(name)
    if os.path.exists(path):
        n = len(open(path).read().splitlines())
        print(f"You have a {name}, {n} lines long. Open it with an editor to see it.")
        break
else:
    print("Neither ~/.zshrc nor ~/.bashrc exists on this machine.")

## Your first shell script

A script is just a file with commands in it.

1. The first line, the **shebang**, says which interpreter to use.
2. Make it executable with `chmod +x`.
3. Run it with `./script.sh`.

Note `./`: the shell does **not** look in the current directory unless you say so.

In [ ]:
%%writefile hello.sh
#!/bin/bash
# A first script
echo "Hello from $(basename $0)"
echo "Today is $(date +%Y-%m-%d)"
echo "There are $(ls | wc -l) entries in $(basename $(pwd))/"

In [ ]:
!chmod +x hello.sh
!./hello.sh

## Cleaning up

Everything we made lives in `unix_sandbox`. This removes it.

> Run this **only when you are done**. Afterwards the sandbox is gone, and the
> cells above will fail until you re-run the setup cell that created it.

In [ ]:
import os, shutil

BASE = globals().get("BASE", os.getcwd())      # also works after a kernel restart
os.chdir(BASE)
shutil.rmtree("unix_sandbox", ignore_errors=True)
print("Sandbox removed. Back in:", os.path.basename(os.getcwd()) + "/")

## Exercises

**1.** Create a directory `FCA`, move into it, create a file `README.md` containing
the line `# Advanced Computational Physics`, and show its contents.

**2.** Create ten files `run01.txt` ... `run10.txt` in a single command.
*Hint: `touch run{01..10}.txt`.*

**3.** Count how many lines in `/etc/passwd` contain the string `bin`, using a pipe.

**4.** What is wrong with each of these?

```bash
cd data/                 # and then the next command says: No such file or directory
rm *.txt file.txt
./script.sh              # Permission denied
echo "result" > out.txt  # run twice, and only one line is there
```

**5.** Write a script `count.sh` that takes a directory as its argument and prints
how many `.txt` files it contains.

## Cheat sheet

| | |
|---|---|
| `pwd` `ls` `cd` | where am I, what is here, go there |
| `mkdir` `touch` `rm` `rmdir` | create and destroy |
| `cp` `mv` | copy, move/rename |
| `cat` `head` `tail` `less` `wc` | look inside |
| `*` `?` `[ ]` | wildcards |
| `>` `>>` `<` `\|` | redirection and pipes |
| `grep` `find` | search inside files / for files |
| `chmod` `ls -l` | permissions |
| `echo $VAR` `export` `which` | environment |
| `ps` `kill` `Ctrl-C` `&` | processes |
| `ssh` `scp` `screen` | remote work |
| `man` `--help` | when in doubt |